Smart Routing — The Right Model for the Right Job

In [23]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [24]:
model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "ollama/gemma4:31b-cloud",
            "api_base": "http://127.0.0.1:11434"
        }
    },
    {
        "model_name": "smart-coding",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b"
        }
    }
]

In [25]:
from litellm import Router


router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[
        {
            "role": "user",
            "content": "Summarize: AI is changing software."
        }
    ]
)

code_response = router.completion(
    model="smart-coding",
    messages=[
        {
            "role": "user",
            "content": "Write a Python function to reverse a string."
        }
    ]
)

print(
    "Fast/cheap (Ollama Gemma):",
    fast_response.choices[0].message.content[:150]
)

print(
    "\nSmart/coding (Groq):",
    code_response.choices[0].message.content
)

Fast/cheap (Ollama Gemma): AI is transforming software development and functionality by automating coding tasks, enabling smarter features, and shifting how applications are bui

Smart/coding (Groq): Here’s a tiny, self‑contained Python function that reverses any string you give it:

```python
def reverse_string(s: str) -> str:
    """
    Return a new string containing the characters of `s` in reverse order.

    Parameters
    ----------
    s : str
        The string to reverse.

    Returns
    -------
    str
        The reversed string.
    """
    # The most idiomatic way in CPython is the slice‑step trick:
    return s[::-1]
```

### How it works
- `s[::-1]` creates a slice that starts at the end of the string (`:`) and steps backwards by `-1`, effectively reading the string from right to left.

### Example usage

```python
>>> reverse_string("hello")
'olleh'

>>> reverse_string("Python 3.8")
'8.3 nohtyP'

>>> reverse_string("")
''
```

### A more manual (but educational) imple

Load Balancing Across Multiple API Keys

In [27]:
# simple shuffle

from litellm import Router

model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
                    "model": "ollama/gemma4:31b-cloud",
                    "api_base": "http://127.0.0.1:11434"
                },
        "model_info": {
            "id": "ollama-cloud"
        }
    },
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
        },
        "model_info": {
            "id": "groq-gpt-oss-20b"
        }
    },
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[
            {
                "role": "user",
                "content": f"Say hello, request {i + 1}"
            }
        ]
    )

    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content

    print(
        f"#{i + 1:<9}"
        f"{deployment_id:<22}"
        f"{latency:>6.0f} ms   "
        f"{answer}"
    )

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        ollama-cloud             658 ms   Hello! I am ready for request 1. Please let me know what you need.
#2        ollama-cloud            1521 ms   Hello! Please provide your 2 requests.
#3        groq-gpt-oss-20b        1319 ms   Hello! request 3
#4        ollama-cloud            1627 ms   Hello! Please provide request 4.
#5        ollama-cloud             709 ms   Hello! Please provide 5 items, topics, or requests.
#6        groq-gpt-oss-20b         911 ms   Hello! Could you please provide the 6?


In [28]:
# least busy

from litellm import Router


router = Router(
    model_list=model_list,
    routing_strategy="least-busy"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[
            {
                "role": "user",
                "content": f"Say hello, request {i + 1}"
            }
        ]
    )

    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]

    print(
        f"#{i + 1:<9}"
        f"{deployment_id:<22}"
        f"{latency:>6.0f} ms   "
        f"{answer}"
    )

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        ollama-cloud             777 ms   Hello! Please let me know what your
#2        ollama-cloud             696 ms   Hello! I'm ready. What is your seco
#3        ollama-cloud             575 ms   Hello! Please provide your 3 reques
#4        ollama-cloud             638 ms   Hello! Please provide request 4.
#5        ollama-cloud             823 ms   Hello! Please provide 5 of whatever
#6        ollama-cloud             701 ms   Hello! Please provide request 6.


In [29]:
# latency based routing

from litellm import Router



router = Router(
    model_list=model_list,
    routing_strategy="latency-based-routing"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-" * 84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[
            {
                "role": "user",
                "content": f"Say hello, request {i + 1}"
            }
        ]
    )

    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content

    print(
        f"#{i + 1:<9}"
        f"{deployment_id:<22}"
        f"{latency:>6.0f} ms   "
        f"{answer}"
    )

Request   Deployment Picked     Latency     Response                                
------------------------------------------------------------------------------------
#1        groq-gpt-oss-20b         518 ms   Hello!
#2        groq-gpt-oss-20b         901 ms   Hello! 👋

I’m ready to help with your second request—could you let me know what you’d like me to do?
#3        groq-gpt-oss-20b         507 ms   Hello!  
Could you please share three things you’d like to talk about or need help with?
#4        ollama-cloud            1423 ms   Hello! Please provide the details for request 4.
#5        ollama-cloud             700 ms   Hello! Please provide 5 items, topics, or requests you would like me to handle.
#6        groq-gpt-oss-20b         811 ms   Hello! Could you clarify what you mean by “request 6”?


| Strategy                | How it works                                          |
| ----------------------- | ----------------------------------------------------- |
| `simple-shuffle`        | Randomly/shuffle selects between deployments          |
| `least-busy`            | Selects the deployment with the least active requests |
| `latency-based-routing` | Routes toward deployments with lower observed latency |
| `usage-based-routing`   | Routes based on current usage                         |
| `cost-based-routing`    | Routes based on cost                                  |
| `lowest-latency`        | Selects the deployment with the lowest latency        |
| `weighted`              | Gives deployments different probabilities/weights     |
| `rate-limit-aware`      | Considers rate-limit information when routing         |
